# Installing important libraries

In [1]:
%pip install  openai langchain faiss-cpu pypdf tiktoken docarray PyPDF tiktoken langchain-openai flashrank langchain-community pillow sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


In [3]:
load_dotenv()

True

In [4]:
# Load PDF documents with error handling
print("Loading PDF documents from ./Policy+Documents...")
try:
    pdf_directory_loader = PyPDFDirectoryLoader("./Policy+Documents")
    documents = pdf_directory_loader.load()
    print(f"✓ Successfully loaded {len(documents)} documents")
    print(f"✓ Total pages: {sum(doc.metadata.get('total_pages', 1) for doc in documents)}")
except Exception as e:
    print(f"Error loading documents: {str(e)}")
    raise

Loading PDF documents from ./Policy+Documents...
✓ Successfully loaded 217 documents
✓ Total pages: 7209
✓ Successfully loaded 217 documents
✓ Total pages: 7209


In [5]:
documents[0].page_content[:100]

'Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s Contact Num'

In [6]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 760 document chunks
✓ Average chunk size: 848 characters


In [7]:
print(splits[0])

page_content='Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Your Policy no. <<  >> 
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health (“Policy”) 
being this document, has been issued. We have made every effort to design your Policy in a simple format. We 
have highlighted items of importance so that you may recognize them easily. 
 
Policy document: 
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy 
is enclosed herewith. Please preserve this document safely and also inform your nominees about the same. A 
copy of your proposal form and other relevant documents submitted by you is also enclosed for your 
information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the Policy, you have the option to' metad

In [8]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized
✓ Embeddings model initialized


In [9]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...
✓ Test embedding successful - dimension: 1536
✓ Test embedding successful - dimension: 1536


In [10]:
from langchain_classic.embeddings import CacheBackedEmbeddings  
from langchain_classic.storage import LocalFileStore 
store = LocalFileStore("./cache/") 

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model,
    store,
    namespace="semantic-spotter"
)

c:\Python311\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [11]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyholder’s Name>>,  
 
Sub: Yo...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: information and record.  
 
Cancellation in the Free-Look Period: 
 
<< In case you are not agreeable to any of the terms and conditions stated in the...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: 0
Content preview: option to return the Policy to us for cancellation stating the reasons thereof, within 15 days from the date of 
receipt of the Policy. On receipt of ...

✓ Total splits available: 760


In [12]:

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """Create and save FAISS vector store."""
    print(f"Creating vector store from {len(splits)} documents...")
    start_time = time.time()
    
    try:
        # Create FAISS store directly from documents
        if os.path.exists(save_path):
            return FAISS.load_local(save_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)
        
        vectordb = FAISS.from_documents(
            documents=splits,
            embedding=embeddings_model
        )
        print(f"✓ FAISS vector store created")
        
        # Save to disk
        os.makedirs(save_path, exist_ok=True)
        vectordb.save_local(save_path)
        print(f"✓ Saved to: {save_path}")
        
        elapsed = time.time() - start_time
        print(f"✓ Time: {elapsed:.1f}s ({elapsed/60:.1f}m)")
        
        return vectordb
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {e}")
        raise


In [15]:
# Create the vector store
try:
    vectordb = create_vector_store_faiss(splits, cached_embedder, "./faiss_store")
    print("✓ Vector store ready for similarity search")
except Exception as e:
    print(f"Failed to create vector store: {str(e)}")
    raise


Creating vector store from 760 documents...
✓ FAISS vector store created
✓ Saved to: ./faiss_store
✓ Time: 14.1s (0.2m)
✓ Vector store ready for similarity search
✓ FAISS vector store created
✓ Saved to: ./faiss_store
✓ Time: 14.1s (0.2m)
✓ Vector store ready for similarity search


In [17]:
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = None

if 'vectordb' in globals() and vectordb is not None:
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
    )

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""


    if 'vectordb' not in globals() and vectordb is None:
        return "No vector store available", []
    
    #prefer compression_retriever when available , otherwise use fallback to basic retriever
    try:
        if compression_retriever is not None:
            retrieved_docs = compression_retriever.invoke(
            query
        )
            
        else:
            retrieved_docs = vectordb.as_retriever(search_kwargs={"k": 5}).get_relevant_documents(query)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    # serialized = "\n\n".join(
    #     (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
    #     for doc in retrieved_docs
    # )
       
    except Exception as e:
            print("error in retrieve_context",e)

    serialized = "\n\n".join(
            f"Source: {d.metadata.get('source','unknown')} | Page: {d.metadata.get('page','?')}\n{d.page_content}"
            for d in retrieved_docs
        )
    return serialized, retrieved_docs

In [18]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

# Instantiate the LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", streaming=True)

tools = [retrieve_context]

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]
2. Do NOT include any explanation, preamble, or extra text
3. Do NOT repeat the question
4. Do NOT include metadata (producer, creator, page, author, etc.)
5. Use only the actual policy content
6. If answer not found, write: "Not found in the provided policy context."

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

prompt = """You are an insurance policy QA assistant. Answer questions ONLY based on the policy documents provided in details.

CRITICAL OUTPUT FORMAT RULES:
1. ALWAYS respond with EXACTLY this format, nothing else:
answer: [your answer here]
source: [metadata info from retrieved document pdf name and page no.]

EXAMPLE OUTPUT:
answer: The policy covers up to $500,000 for life insurance
"""

agent = create_agent(llm, tools, system_prompt=prompt)


In [19]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    response = agent.invoke({
        'messages': [
            HumanMessage(content=(
            query
            ))
        ]
    })

    print(response['messages'][-1].content)

In [20]:
insurance_agent( "What is the life insurance policy coverage amount?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The maximum benefit payable on Accidental Death of an Insured Member under all policies is limited to a total of Rs.10,000,000/- (Rupees 1 crore only). 
source: Policy+Documents\HDFC-Life-Group-Term-Life-Policy.pdf | Page: 10


In [21]:
insurance_agent( "Can a 100 year plus person do a term insurance?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: A person aged 100 years or older is not eligible for term insurance, as there are specified age limits for entry and coverage that do not accommodate this age group. 
source: Policy+Documents\HDFC-Life-Group-Term-Life-Policy.pdf | Page: 13


In [22]:
insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The definitions of covered Critical Illnesses include Myocardial Infarction (First Heart Attack of specific severity), which is the first occurrence of a heart attack evidenced by typical clinical symptoms, specific electrocardiogram changes, and elevation of infarction-specific enzymes. Certain exclusions apply, such as other acute coronary syndromes and types of angina pectoris. Additionally, benefits are not payable for claims made within specified time frames after policy commencement or death following diagnosis.
source: Policy+Documents\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: 26


In [23]:
# retrieve_context("what is the Definitions of Critical Illnesses? based on policy?")

In [24]:
# retrieve_context("Can a 100 year plus person do a term insurance?")

In [25]:
insurance_agent("what is the life insurance coverage for disability?")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The policy does not explicitly mention disability coverage benefits under life insurance; it focuses more on the death benefits and critical illness coverage. 
source: Policy+Documents\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: 7
